Step 1: Install

In [ ]:
!pip install --quiet anthropic pydantic

Step 2: Load key from Collab Secrets, create client

In [ ]:
from google.colab import userdata
import anthropic
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Client()
print("Client ready to use!")

Step 3: Message call invocation

In [ ]:
from anthropic import APIStatusError, BadRequestError

MODEL = "claude-haiku-4-5-20251001"
MAX_TOKENS = 512

try:
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "user", "content": "In one sentence, what is a Security Operations Center?"}
        ]
    )
    print("API call successful:")
    print(response.content[0].text)
except BadRequestError as e:
    print(f"Bad Request Error: {e.response.json()['error']['message']}")
    print("Please check your API request parameters or your Anthropic account's billing status.")
except APIStatusError as e:
    print(f"An API error occurred: {e.status_code} - {e.response.json()['error']['message']}")
    print("You might need to check your internet connection or the Anthropic API service status.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Step 4: Understand the response

In [ ]:
print("model:", response.model)
print("stop reason:", response.stop_reason)
print("input tokens:", response.usage.input_tokens)
print("output tokens:", response.usage.output_tokens)
print("-" * 40)

for i, block in enumerate(response):
    if block[0] == "content":
        print(i, block)
        print("-" * 40)

Step 5: Helper function

In [ ]:
def ask(prompt: str, system: str | None = None, max_tokens: int = 512) -> str:
    """Send one user message to Claude and return the concatenated text."""
    kwargs = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        kwargs["system"] = system
    response = None
    try:
        response = client.messages.create(**kwargs)
        print(response)
    except anthropic.APIStatusError as e:
        print(f"An API error occurred: {e.status_code} - {e.response.json()['error']['message']}")
        print("You might need to check your internet connection or the Anthropic API service status.")
    result = "".join(r.text for r in response.content if r.type == "text")
    return result.replace("\n", "")

# test helper function
ask("Give one liner definition of MITRE ATT&CK.")